# Breakout 2B · Trade and the border: mirrors, segments, networks, regimes

Workshop **Data Driven Domestic Revenue Mobilization**, Bangkok, October 2026.

Mirror data read as a series, traders as segments, brokers as a scorecard, the network as a
picture and the three regimes as totals. The exercise is the 24 mirror gaps of the worksheet
and Country A's chapters on the regional map.

Press ▶ on each cell from top to bottom. Change only the sliders and drop-down menus. Everything
here is synthetic; Country A to J are anonymised neighbours.

In [ ]:
#@title 0 · Where the data comes from
#@markdown Leave these as they are. Your analysts can point DATA_URL at their own extract (section 6).
DATA_URL = "https://raw.githubusercontent.com/FrancoisChastel/CustomsWorkshopNotebooks/data-v6/data"  #@param {type:"string"}
INSTALL = True  #@param {type:"boolean"}

In [ ]:
#@title 0 · Setup (run me first)
import sys, subprocess, warnings
warnings.filterwarnings("ignore")
if INSTALL and "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/FrancoisChastel/CustomsWorkshopNotebooks@data-v6"], check=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from workshop_methods import notebook as nb
from workshop_methods.data import load_json, load_tables
from workshop_methods import border, trade, valuation
nb.use_style()
pd.set_option("display.max_colwidth", 40)
T = load_tables(DATA_URL)
print(f"Loaded {len(T)} tables, {len(T['declarations']):,} declaration lines. Ready.")

In [ ]:
#@title 0 · Prepare the tables
d, traders = T["declarations"], T["traders"]
mirror, ruv, sector = T["mirror_trade"], T["regional_unit_values"], T["sector_regional"]
name = traders.set_index("trader_ref")["trader_id"]
name = name.where(name != "", traders.set_index("trader_ref").index.to_series())
latest = int(mirror["year"].max())
rated = trade.mirror_ratios(mirror)
print(f"Mirror data for {mirror['chapter'].nunique()} chapters x {mirror['partner'].nunique()} partners, "
      f"{mirror['year'].min()} to {latest}; regional prices for {ruv['importer'].nunique()} countries.")

## 1 · Mirror data, two ways

The partner's export statistics are a second witness of what crossed the border. Our customs value
includes shipping and insurance (CIF), the partner's figure does not (FOB), so an honest corridor
sits a little above 1.

In [ ]:
#@title 1 · One corridor over five years
CHAPTER = "17"  #@param ["17","24","39","52","61","72","85","87"]
PARTNER = "Thailand"  #@param ["Thailand","Viet Nam","China"]
nb.steps("Does home record what the partner says it shipped?",
         "Divide home imports by the partner's exports, year by year.",
         "Compare with 1 + the shipping-and-insurance margin, where an honest corridor sits.")
series = rated[(rated["chapter"] == CHAPTER) & (rated["partner"] == PARTNER)].sort_values("year")
honest = 1 + series["cif_fob_adj"].iloc[-1]
fig, ax = nb.chart(f"Chapter {CHAPTER} from {PARTNER}: home imports against the partner's exports")
ax.plot(series["year"], series["ratio"], marker="o", color=nb.COLORS["accent"], label="home imports ÷ partner exports")
ax.axhline(honest, linestyle="--", color=nb.COLORS["muted"], label=f"honest corridor, about {honest:.2f}")
ax.set_xticks(series["year"]); ax.set_ylim(0, max(1.4, series["ratio"].max() * 1.1)); ax.legend(); plt.show()
last = series.iloc[-1]
if np.isnan(last["ratio"]):
    nb.finding(f"{PARTNER} did not report its {latest} exports in chapter {CHAPTER}: the mirror needs better data, not a query.")
else:
    nb.finding(f"In {latest} home recorded {last['ratio']:.0%} of what {PARTNER} reports shipping in chapter {CHAPTER}; "
               f"an honest corridor sits near {honest:.2f}.")

In [ ]:
#@title 1 · The bridge: subtract each known explanation
CIF_FOB = 0.08  #@param {type:"slider", min:0, max:0.15, step:0.01}
TIMING = True  #@param {type:"boolean"}
HUB_SHARE = 1.0  #@param {type:"slider", min:0, max:1, step:0.25}
nb.steps("How much of the sugar gap from Thailand survives every known explanation?",
         "Start from the partner's exports and add shipping and insurance: the expected home imports.",
         "Add back goods in transit at the year end (timing) and goods attributed to the hub that re-ships them.",
         "What is left is the residual worth a query.")
row = mirror[(mirror["chapter"] == "17") & (mirror["partner"] == "Thailand") & (mirror["year"] == latest)].iloc[0]
b = trade.bridge(row["partner_exports_usd"] / 1e6, row["home_imports_usd"] / 1e6, CIF_FOB,
                 row["timing_usd"] / 1e6 if TIMING else 0.0, row["hub_attributed_usd"] / 1e6 * HUB_SHARE)
steps_ = {"partner exports": b["partner_exports"], "expected at home": b["expected_cif"], "recorded": b["recorded"],
          "+ timing": b["timing"], "+ hub": b["hub"], "residual": b["residual"]}
fig, ax = nb.chart(f"Sugar from Thailand, {latest}: {b['residual_share']:.0%} of expected imports unexplained")
colors = [nb.COLORS["muted"], nb.COLORS["navy2"], nb.COLORS["accent"], nb.COLORS["pale"], nb.COLORS["pale"], nb.COLORS["amber"]]
ax.bar(list(steps_), list(steps_.values()), color=colors)
ax.set_ylabel("USD million"); plt.show()
nb.finding(f"Partner exports of USD {b['partner_exports']:.1f} million should arrive as {b['expected_cif']:.1f} million with shipping "
           f"and insurance; recorded {b['recorded']:.1f} + timing {b['timing']:.1f} + hub {b['hub']:.1f} explain {b['explained']:.1f}. "
           f"The residual, {b['residual']:.1f} million or {b['residual_share']:.0%}, is a query, not a verdict.")

In [ ]:
#@title 1b · Regional prices: what neighbours pay for the same goods
EXPORTER = "all sellers"  #@param ["all sellers", "Australia", "China", "India", "Indonesia", "Japan", "Korea", "Malaysia", "Singapore", "Thailand", "United Arab Emirates", "Viet Nam"]

THRESHOLD = 0.70  #@param {type:"slider", min:0.5, max:0.9, step:0.05}
nb.steps("Does Country A pay much less than its neighbours for the same goods from the same seller?",
         "For each good, seller and year, divide each country's value per kilo by the median of the ten.",
         f"Flag Country A where it pays less than {THRESHOLD:.0%} of the median; check how much of the value has a weight.")
part = ruv if EXPORTER == "all sellers" else ruv[ruv["exporter"] == EXPORTER]
flags = trade.regional_flags(part, threshold=THRESHOLD)
dots = part.groupby(["commodity", "importer"])["uv_ratio"].mean().unstack()
order = flags.sort_values("ratio", ascending=False)["commodity"].tolist()
fig, ax = nb.chart(f"Country A against the regional median ({EXPORTER})")
for i, commodity in enumerate(order):
    others = dots.loc[commodity].drop("Country A")
    ax.scatter(others, [i] * len(others), color=nb.COLORS["muted"], s=18)
    a = dots.at[commodity, "Country A"]
    ax.scatter([a], [i], color=nb.COLORS["amber"] if a < THRESHOLD else nb.COLORS["accent"], s=70, zorder=3)
ax.axvline(1.0, color=nb.COLORS["ink"], linewidth=1); ax.axvline(THRESHOLD, color=nb.COLORS["amber"], linestyle=":")
ax.set_yticks(range(len(order)), [c.replace("_", " ") for c in order]); ax.set_xlabel("ratio to the regional median"); plt.show()
low = flags[flags["below_threshold"]]
trusted = low[low["trusted"]]
doubtful = low[~low["trusted"]]
text = ", ".join(f"{c.replace('_', ' ')} ({r:.2f})" for c, r in zip(trusted["commodity"], trusted["ratio"]))
extra = "".join(f" {c.replace('_', ' ')} ({r:.2f}) is low too, but weights cover only {w:.0%} of its value." for c, r, w in
                zip(doubtful["commodity"], doubtful["ratio"], doubtful["weight_coverage"]))
nb.finding(f"Country A pays under {THRESHOLD:.0%} of the regional median for {text or 'no good'}.{extra}")

In [ ]:
#@title 1c · Risky sectors: the chapter by country map
YEAR = 2025  #@param [2023, 2024, 2025]
nb.steps("Where in the region are prices low, by chapter?",
         "Value-weighted ratio to the regional median, chapter by country.",
         "Read a column: one country's audit map. Read a row: a sector worth a regional conversation.")
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
grid = trade.sector_map(sector, YEAR)
cmap = LinearSegmentedColormap.from_list("ws", [nb.COLORS["amber"], "#FFFFFF", nb.COLORS["accent"]])
fig, ax = nb.chart(f"{YEAR}: chapters below the regional median are amber", height=4.8)
ax.imshow(grid.to_numpy(), cmap=cmap, norm=TwoSlopeNorm(vcenter=1.0, vmin=0.5, vmax=1.5), aspect="auto")
ax.set_xticks(range(grid.shape[1]), [c.replace("Country ", "") for c in grid.columns])
ax.set_yticks(range(grid.shape[0]), [f"ch. {c}" for c in grid.index]); ax.grid(False)
for (i, j), v in np.ndenumerate(grid.to_numpy()):
    if v < 0.70:
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8, color=nb.COLORS["navy"])
ax.set_xlabel("country (A to J)"); plt.show()
column = grid["Country A"]
low_a = column[column < 0.70]
elsewhere = grid.drop(columns="Country A").stack()
elsewhere = elsewhere[elsewhere < 0.70]
listed = [f"{c} ({v:.2f})" for c, v in low_a.items()]
listed = ", ".join(listed[:-1]) + " and " + listed[-1] if len(listed) > 1 else "".join(listed)
nb.finding(f"Country A's column is low in chapters {listed}; "
           f"{len(elsewhere)} other cells are below 0.70 ({', '.join(f'ch. {c} in {k}' for (c, k) in elsewhere.index[:4])}).")

## 1d · Valuation: what similar goods were accepted at

For one invoice line, the accepted lines of the same goods and origin whose descriptions are most alike, with
every price put on one basis (US dollars, FOB). The range is a reference for a question, never a value.

In [ ]:
#@title 1d · The neighbours of one invoice line
LINE = "V2"  #@param ["V1", "V2", "V3", "V4", "V5"]
FLOOR = 0.85  #@param {type:"slider", min:0.7, max:0.95, step:0.01}
WINDOW_DAYS = 90  #@param {type:"slider", min:30, max:90, step:15}
K = 30  #@param {type:"slider", min:10, max:50, step:5}
history, invoice = T["accepted_history"], T["invoice_lines"]
similarity = load_json(DATA_URL, "valuation_similarity")["lines"]
line = invoice.set_index("ref").loc[LINE]
nb.steps(f"What were goods like '{line['description']}' from {line['origin']} accepted at?",
         f"Keep accepted lines of the same heading ({line['hs6']}) and origin from the last {WINDOW_DAYS} days.",
         f"Keep those whose description is at least {FLOOR:.2f} alike (multilingual sentence embeddings), the {K} closest.",
         "Convert every price to US dollars, FOB, net of discounts; read the median and the middle half.")
kept = valuation.neighbours(line, history, similarity[LINE], T["fx_rates"], k=K, floor=FLOOR, window_days=WINDOW_DAYS)
rng = valuation.price_range(kept)
declared = float(valuation.to_fob_usd(invoice[invoice["ref"] == LINE], T["fx_rates"], "invoice_date").iloc[0])
verdict = valuation.assess(declared, rng)
nb.show(kept[["accepted_on", "description", "currency", "incoterm", "unit_price", "fob_usd", "similarity"]].head(12),
        formats={"unit_price": "{:,.2f}", "fob_usd": "{:,.2f}", "similarity": "{:.2f}"})
if rng["count"]:
    fig, ax = nb.chart(f"Line {LINE}: declared {declared:,.2f} against {rng['count']} neighbours")
    ax.hist(kept["fob_usd"], bins=12, color=nb.COLORS["pale"])
    ax.axvspan(rng["p25"], rng["p75"], color=nb.COLORS["accent"], alpha=0.15, label="middle half")
    ax.axvline(rng["median"], color=nb.COLORS["accent"], label=f"median {rng['median']:,.2f}")
    ax.axvline(declared, color=nb.COLORS["amber"], linewidth=2.5, label=f"declared {declared:,.2f}")
    ax.set_xlabel("unit price, USD FOB"); ax.legend(); plt.show()
    nb.finding(f"{rng['count']} neighbours, median {rng['median']:,.2f}, middle half {rng['p25']:,.2f} to {rng['p75']:,.2f}; "
               f"declared {declared:,.2f} = {verdict['share_of_median']:.0%} of the median, {verdict['distance']:.1f} spreads. "
               f"Decision: {verdict['decision']} ({verdict['reason']}).")
else:
    nb.finding(f"No accepted line of the same goods and origin passes a floor of {FLOOR:.2f}: there is no range to read.")

## 2 · Reading a series

In [ ]:
#@title 2 · Persistent: every year below 0.80 · growing: falling and ending below · spike: one year 30 points below the rest
nb.steps("What shape does each corridor's gap have?",
         "Use the ratio after shipping, timing and hubs, year by year.",
         "Persistent: every year below 0.80. Growing: falling more than 5 points a year and ending below 0.80. "
         "Spike: one year at least 30 points below the others.")
shapes = trade.classify_series(mirror)
flagged = shapes[shapes["shape"].isin(["persistent", "growing", "spike"])]
examples = flagged.drop_duplicates("shape").set_index("shape").loc[[s for s in ["persistent", "growing", "spike"] if s in set(flagged["shape"])]]
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(examples), figsize=(8.6, 3.2), sharey=True)
fig.suptitle("One corridor of each shape", x=0.01, ha="left", color=nb.COLORS["navy"], fontweight="bold")
for ax, (shape, ex) in zip(np.atleast_1d(axes), examples.iterrows()):
    s = rated[(rated["chapter"] == ex["chapter"]) & (rated["partner"] == ex["partner"])].sort_values("year")
    ax.plot(s["year"], s["ratio_after_known"], marker="o", color=nb.COLORS["amber"])
    ax.axhline(0.80, color=nb.COLORS["muted"], linestyle=":"); ax.set_title(f"{shape}: ch. {ex['chapter']}, {ex['partner']}", fontsize=10)
    ax.set_xticks(s["year"][::2])
plt.show()
nb.show(flagged[["chapter", "partner", "shape", "mean_ratio", "years_reported"]].reset_index(drop=True), formats={"mean_ratio": "{:.2f}"})
words = {s: ", ".join(f"ch. {c} {p}" for c, p in zip(g["chapter"], g["partner"])) for s, g in flagged.groupby("shape")}
nb.finding("; ".join(f"{len(flagged[flagged['shape'] == s])} {s} ({w})" for s, w in words.items())
           + f". The other {len(shapes) - len(flagged)} corridors look normal once the known explanations are applied.")

## 3 · Segments and brokers

In [ ]:
#@title 3 · Four tiles of large importers, and what the broker adds
ADD_BROKER = True  #@param {type:"boolean"}
nb.steps("Which large importers deserve a closer look, and does their broker change the answer?",
         "For the 100 largest importers: 12-month import value and a compliance score (findings per inspection, amendments, queries upheld).",
         "Group them into four tiles (k-means).", "Add a third signal: the share of the broker's clients that had a finding.")
profile = border.trader_profile(d)
big = profile.nlargest(100, "import_value_12m")
scorecard = border.broker_scorecard(d)
before = border.segments(big)
after = border.segments(big, broker_share=scorecard["flagged_client_share"])
seg = after if ADD_BROKER else before
rank_before = before["risk_axis"].rank(ascending=False)
rank_after = after["risk_axis"].rank(ascending=False)
entered = (rank_after <= 10) & (rank_before > 10)
mover = (rank_before - rank_after)[entered].idxmax() if entered.any() else None
centres = seg.groupby("segment").agg(value=("import_value_12m", "median"), risk=("risk_axis", "mean"))
tile = {s: f"{'larger' if r['value'] >= centres['value'].median() else 'smaller'}, {'watch' if r['risk'] >= centres['risk'].median() else 'low risk'}"
        for s, r in centres.iterrows()}
fig, ax = nb.chart("Large importers by value and risk" + (" (broker signal added)" if ADD_BROKER else ""))
palette = [nb.COLORS["accent"], nb.COLORS["muted"], nb.COLORS["pale"], nb.COLORS["navy2"]]
for s, part in seg.groupby("segment"):
    ax.scatter(part["import_value_12m"] / 1e6, part["risk_axis"], s=24, color=palette[s % 4], label=tile[s])
if mover is not None:
    ax.scatter(seg.at[mover, "import_value_12m"] / 1e6, seg.at[mover, "risk_axis"], s=220, facecolors="none",
               edgecolors=nb.COLORS["amber"], linewidths=2.5, label="moved with the broker signal")
ax.set_xscale("log"); ax.set_xlabel("imports, 12 months, LCU million (log scale)"); ax.set_ylabel("risk (compliance + broker)")
ax.legend(fontsize=8); plt.show()
if mover is not None:
    nb.finding(f"Adding the broker signal moves importer {name[mover]} from #{rank_before[mover]:.0f} to #{rank_after[mover]:.0f} "
               f"of the 100 largest: its own record is clean, its broker ({profile.at[mover, 'main_broker']}) serves many importers with findings.")
else:
    nb.finding("No large importer changes its place much when the broker signal is added.")

In [ ]:
#@title 3 · The broker scorecard
nb.steps("Which brokers serve many importers with findings?",
         "For each broker with five clients or more: clients, lines, the share of clients with a finding,",
         "amendment rate, queries upheld and release time, against the median broker.")
view = scorecard[scorecard["clients"] >= 5].head(8)[["clients", "lines", "flagged_client_share", "vs_peer_median",
                                                        "amendment_rate", "queries_upheld_rate", "mean_release_hours"]]
nb.show(view, formats={"flagged_client_share": "{:.0%}", "vs_peer_median": "{:+.0%}", "amendment_rate": "{:.1%}",
                       "queries_upheld_rate": "{:.1%}", "mean_release_hours": "{:.1f}"})
top = view.iloc[0]
nb.finding(f"Broker {view.index[0]}: {top['flagged_client_share']:.0%} of its {top['clients']:.0f} clients had a finding, "
           f"{top['vs_peer_median']:+.0%} against the median broker. One broker can lodge for many risky importers.")

## 4 · Networks and regimes

In [ ]:
#@title 4 · The network: who sits in the middle?
nb.steps("Which sellers connect to the most importers?",
         "Draw importers, sellers, brokers, consignees and addresses as dots, and every shared field as a line.",
         "Count each seller's links; the projector game is to find the hub before the table does.")
net = load_json(DATA_URL, "network")
G = border.build_graph(net)
top = border.hubs(G, kind="exporter", top=10)
kinds = {"trader": nb.COLORS["accent"], "exporter": nb.COLORS["amber"], "broker": nb.COLORS["navy2"],
         "consignee": nb.COLORS["pale"], "address": nb.COLORS["muted"]}
sub = G.subgraph(set(top["node"]) | {n for x in top["node"] for n in G.neighbors(x)})
try:
    from pyvis.network import Network
    from IPython.display import HTML, display
    view = Network(height="520px", width="100%", notebook=True, cdn_resources="remote")
    for n, data in sub.nodes(data=True):
        view.add_node(n, label=n, color=kinds.get(data["kind"], nb.COLORS["muted"]), size=6 + 1.5 * sub.degree(n), title=data["kind"])
    for a, b2 in sub.edges():
        view.add_edge(a, b2, color=nb.COLORS["border"])
    display(HTML(view.generate_html()))
except ImportError:
    import networkx as nx
    fig, ax = nb.chart("Sellers and their importers")
    pos = nx.spring_layout(sub, seed=1)
    nx.draw_networkx(sub, pos, ax=ax, node_size=[10 + 4 * sub.degree(n) for n in sub], with_labels=False,
                     node_color=[kinds.get(sub.nodes[n]["kind"], nb.COLORS["muted"]) for n in sub], edge_color=nb.COLORS["border"])
    plt.show()
nb.show(top)
median = np.median([G.degree(n) for n, k in G.nodes(data="kind") if k == "exporter"])
nb.finding(f"Sellers {top['node'][0]} and {top['node'][1]} connect to {top['degree'][0]} and {top['degree'][1]} importers, "
           f"against a median of {median:.0f} for all sellers.")

In [ ]:
#@title 4 · Open one node
NODE = "X2012"  #@param ["X2012", "X2015", "X2213", "X2136", "X2144", "X2059", "X2139", "X2031", "X2044", "X2222"]

nb.steps("What does this seller's trade look like?", "Take the import lines it sold.", "Group them by importer.")
lines = d[(d["exporter_id"] == NODE) & (d["flow"] == "import")]
by = lines.groupby("trader_ref").agg(lines=("decl_id", "size"), value_lcu=("customs_value_lcu", "sum"),
                                     origin=("origin_country", lambda s: s.mode().iloc[0]),
                                     preference=("preference_claim", lambda s: (s != "none").mean()))
by.index = by.index.map(name)
nb.show(by.sort_values("value_lcu", ascending=False).head(10), formats={"value_lcu": "{:,.0f}", "preference": "{:.0%}"})
nb.finding(f"Seller {NODE} sold to {len(by)} importers; {(lines['preference_claim'] != 'none').mean():.0%} of its lines claim a "
           f"preference, mostly as origin {lines['origin_country'].mode().iloc[0]}.")

In [ ]:
#@title 4 · Regimes: warehouses that do not balance
nb.steps("Does each customs warehouse's stock balance?",
         "Opening stock + goods entered − goods exited − closing stock declared, as a share of goods entered.",
         "Flag months where more than 8% of entries are missing.")
gaps = border.warehouse_gaps(T["warehouse_stock"])
per = gaps.groupby("operator_ref").agg(months=("period", "size"), months_flagged=("flag", "sum"), median_gap=("gap_share", "median"))
per = per.sort_values("median_gap", ascending=False)
per.index = per.index.map(name)
nb.show(per, formats={"median_gap": "{:.1%}"}, highlight=per["months_flagged"] > per["months"] / 2)
leaking = per[per["months_flagged"] > per["months"] / 2]
lo, hi = leaking["median_gap"].min(), leaking["median_gap"].max()
spread = f"{lo:.0%}" if round(lo, 2) == round(hi, 2) else f"{lo:.0%} to {hi:.0%}"
nb.finding(f"{len(leaking)} operators lose more than 8% of entries in most months ({', '.join(leaking.index)}, a median of "
           f"{spread} of entries); the others balance.")

In [ ]:
#@title 4 · Regimes: transit that does not close
nb.steps("Do transit movements close, and on time?",
         "For each operator: the share of movements closed, hours against the route norm, movements open past the deadline.")
perf = border.transit_performance(T["transit"])
perf.index = perf.index.map(name)
nb.show(perf.head(8), formats={"closure_rate": "{:.0%}", "median_hours_vs_norm": "{:.2f}x"})
worst = perf.iloc[0]
nb.finding(f"Operator {perf.index[0]} closes only {worst['closure_rate']:.0%} of {worst['movements']:.0f} movements and leaves "
           f"{worst['open_past_deadline']:.0f} open past the deadline; the median operator closes {perf['closure_rate'].median():.0%}.")

In [ ]:
#@title 4 · Regimes: parcels just under the duty-free limit
DE_MINIMIS = 2000  #@param {type:"integer"}
nb.steps("Does anyone split shipments into parcels just under the duty-free limit?",
         f"For each receiver and month: parcels received and the largest parcel's value against {DE_MINIMIS:,} LCU.",
         "Flag months with six parcels or more, the largest above 90% of the limit.")
split = border.parcel_splitting(T["parcels"], de_minimis=DE_MINIMIS)
nb.show(split.head(8), formats={"max_value_lcu": "{:,.0f}"})
repeat = split[split["months_flagged"] >= 3]
nb.finding(f"{len(repeat)} receivers get six parcels or more a month, each just under the limit, in at least three months: "
           "one shipment split to stay duty free.")

## 5 · Exercise: bin the 24 gaps

Roles: the **analyst** reads the table, the **officer** bins each gap, the **committee** writes the first
check it would sign for every gap it wants investigated. Then tick the chapters where Country A pays more
than 30% below its neighbours. Nothing about the answer is visible until the reveal.

In [ ]:
#@title 5 · The 24 gaps
ws = trade.mirror_ratios(pd.DataFrame(load_json(DATA_URL, "worksheet_2b")["rows"]))
ws["previous_ratio"] = (ws["home_imports_prev_usd"] / (ws["partner_exports_prev_usd"] * (1 + ws["cif_fob_adj"]))).round(3)
ws["gap"] = ws["chapter"] + " · " + ws["partner"]
table = ws.set_index("gap")[["partner_exports_usd", "home_imports_usd", "ratio_cif_adj", "ratio_after_known", "previous_ratio", "note"]]
nb.show(table, formats={"partner_exports_usd": "{:,.0f}", "home_imports_usd": "{:,.0f}", "ratio_cif_adj": "{:.2f}",
                        "ratio_after_known": "{:.2f}", "previous_ratio": "{:.2f}"}, max_rows=24)
low = table[(table["ratio_after_known"] < 0.80) | (table["previous_ratio"] < 0.80)]
nb.finding(f"{len(low)} of the 24 gaps sit below 0.80 this year or last after the known explanations; read the notes before binning.")

In [ ]:
#@title 5 · Bin each gap, tick Country A's chapters, lock
import ipywidgets as widgets
from IPython.display import display
BINS, CHECKS = ["", "explained", "investigate", "needs better data"], ["", "ask the partner for its declarations", "check timing (shipped vs cleared)", "check hub re-exports", "check the weights", "check recoding between codes", "audit the importers"]
regional = pd.DataFrame(load_json(DATA_URL, "worksheet_2b_regional")["cells"])
grid = regional.pivot(index="chapter", columns="importer", values="uv_ratio_to_region")
summary = pd.DataFrame({"Country A": grid["Country A"], "regional median": grid.drop(columns="Country A").median(axis=1),
                        "lowest neighbour": grid.drop(columns="Country A").min(axis=1),
                        "weight coverage A": regional[regional["importer"] == "Country A"].set_index("chapter")["weight_coverage"]})
nb.show(summary, formats={c: "{:.2f}" for c in summary.columns})
rows = [widgets.HBox([widgets.Label(g, layout=widgets.Layout(width="130px")),
                      widgets.Dropdown(options=BINS, layout=widgets.Layout(width="170px")),
                      widgets.Dropdown(options=CHECKS, layout=widgets.Layout(width="270px"))]) for g in table.index]
ticks = [widgets.Checkbox(description=f"chapter {c}", indent=False) for c in summary.index]
lock = widgets.Button(description="Lock my bins and ticks", button_style="primary")
note = widgets.Output()
LOCKED = {"bins": {}, "checks": {}, "chapters": []}
def on_lock(_):
    LOCKED["bins"] = {r.children[0].value: r.children[1].value for r in rows}
    LOCKED["checks"] = {r.children[0].value: r.children[2].value for r in rows}
    LOCKED["chapters"] = [t.description.split()[-1] for t in ticks if t.value]
    with note:
        note.clear_output()
        binned = sum(1 for v in LOCKED["bins"].values() if v)
        print(f"Locked: {binned} of 24 gaps binned, chapters ticked: {', '.join(LOCKED['chapters']) or 'none'}.")
lock.on_click(on_lock)
display(widgets.VBox(rows + [widgets.HBox(ticks), lock, note]))
#@markdown If the menus do not show, type the gaps you would investigate (e.g. "17 · Thailand, 85 · China") and the chapters:
MY_INVESTIGATE = ""  #@param {type:"string"}
MY_CHAPTERS = ""  #@param {type:"string"}
if MY_INVESTIGATE or MY_CHAPTERS:
    LOCKED["bins"] = {g.strip(): "investigate" for g in MY_INVESTIGATE.split(",") if g.strip()}
    LOCKED["chapters"] = [c.strip() for c in MY_CHAPTERS.split(",") if c.strip()]
    print(f"Typed in: investigate {', '.join(LOCKED['bins'])}; chapters {', '.join(LOCKED['chapters'])}.")

In [ ]:
#@title 5 · Track B: the five invoice lines
import ipywidgets as widgets
from IPython.display import display
DECISIONS = ["", "nothing", "indicator", "question"]
invoice = T["invoice_lines"]
rows_b = [widgets.HBox([widgets.Label(f"{r.ref} {r.description[:38]}", layout=widgets.Layout(width="330px")),
                        widgets.Dropdown(options=DECISIONS, layout=widgets.Layout(width="130px")),
                        widgets.Text(placeholder="the reason, in one line", layout=widgets.Layout(width="320px"))])
          for r in invoice.itertuples()]
lock_b = widgets.Button(description="Lock track B", button_style="primary")
note_b = widgets.Output()
LOCKED_B = {}
def on_lock_b(_):
    LOCKED_B.update({r.children[0].value.split()[0]: r.children[1].value for r in rows_b})
    with note_b:
        note_b.clear_output()
        print("Locked: " + ", ".join(f"{k} {v or '-'}" for k, v in LOCKED_B.items()))
lock_b.on_click(on_lock_b)
display(widgets.VBox(rows_b + [lock_b, note_b]))
#@markdown If the menus do not show, type "V1:nothing, V2:question, ..." here instead.
MY_DECISIONS = ""  #@param {type:"string"}
if MY_DECISIONS:
    LOCKED_B.update(dict(p.split(":", 1) for p in MY_DECISIONS.replace(" ", "").split(",") if ":" in p))
    print(f"Typed in: {LOCKED_B}")

## Reveal (facilitator)

The facilitator types the code. Without it, this cell prints nothing.

In [ ]:
#@title 5 · Reveal
REVEAL_CODE = ""  #@param {type:"string"}
SEALED = {"salt": "K/Sb/gRc/nNQ2Rx4Z3SZ8A==", "data": "mm2ZiG16bD9/4upcpBVJR/JeJrfNR0tOTyPfgZNgHyYvsIkR6+fgTsRRxL368xNNDy6OSoIANZpFhr6FAafsBu2MuleRUxHv2LeR4TCwPdsixzCYkFt6pYHCDHgxK0qjtz5an6SqMQJD2WCLUQSq+NWTvAktggpVrcg8GkFFX8jpfnFxY0GUscIuUXOsIiV9WMq8KwTVCfG9bJN+j6/QOS0eMwe2+5jfkdSpLgro34lAz/CNNLM5qKGWob2YiGSaMA26XoItH1mpE5L1vIJyLLy9X/T8qEZT3huuBebRYmY+MYNBcNBemzO7ZRnGyMujjPXTp0LtTfGDAb64i1NCXzgRdVWcK8EAkFHnNXK+34PnEzoac2S7+nYTF46Le3oHwEemIJcqUNHOmBz5fnLvotj9quP7bmwXRf3M4ffiLFgAho0sAhaSQFskHm6Gut1UkUd4yaJNKirn2W37lD3pH6toa4lCHhTgaOKtkZNlH7vNT9byP3uGz9claTEBtIt+NzwHsdTUGsZM/pj3AGgFPZl5L8nnhnmVHTimc0SohhD55BKwgfqEmlv70GgoLRWRk7zdBi15ElNSX3Jl56QHjy/i++A/AJ2IdUceQl1mm9IBkgkV1rqWq+lv+AvN4cRXSclkb0l4TTNpbJ2JC0u0HK8C4tHZ0wU3EGQjuUfJjGTYD1eoZ6ZEgOYhwRDL88thqtVXSjML8dGs80Lm+wmSpFRVRGSaF8aQaRqHvfhU3Sx28l1/SHPHnGEUioX1rZyxmbDbDs3+ig6i8ZTUvuk1QOWBRfNnsmCZwrdn2B6yDZ77Uoh09TgJEnOcb9kSg2e0rhXvvZ55zM9HQaZTHulOXWK/fiJKKeU86PDYnEL0MuZFxWKfKIvMrGQzeyq7IOqVB1r8BgCdSVSPUcGnlh3HiNnI/3dF71+ZfT+4/LK4JL6UT0q9+shiAWe1aEOvOS4J3KXHeewhcpH3fN/PhztChoPKQAF5TwQBwZhwmILSmGVurOHQPnepx31zEd099CeL+Y1vUjfl5PijZBNSmgFZgq28jECesgCcj52t7xG0VNLROb7oEuKl1cWu8GD/zTYncvswBIjxAyM9QEKJyfKzWaGJwWIPafrpY9qkoxxWTuFCh/Raq9ntaEg9OM5psmGAQMTGh93st9U7K+5QBNpPo09m8H9b9rr2eCrvixMfj+HwVjQ+M5pn1F1/M5NWWkvnwFVjOtnh6sfH+VD+IqEHTrHRgp1PTHDdO02Vg5d+d1SBE2GAuFOEyx5b0YSV1TmEPyEwnxK9lDkMrCdqe/02/4AJH23HBTKsVLVUJLOjRiBr8uHE0u1lbg==", "mac": "4d799da33144355c54546ee5dd824610899ceeaf95d3ff4fee393025cdd4c31a"}
answer = nb.unseal(REVEAL_CODE, SEALED)
if answer:
    key = pd.DataFrame(answer["rows"]).assign(gap=lambda f: f["chapter"] + " · " + f["partner"]).set_index("gap")
    key["your bin"] = key.index.map(lambda g: LOCKED["bins"].get(g, ""))
    nb.show(key[["bin", "shape", "your bin", "why"]], max_rows=24, highlight=key["bin"] == "investigate")
    seeded = key[key["bin"] == "investigate"]
    found = int((seeded["your bin"] == "investigate").sum())
    wasted = int(((key["bin"] != "investigate") & (key["your bin"] == "investigate")).sum())
    chapters = set(answer["regional"]["seeded"])
    ticked = set(LOCKED["chapters"])
    nb.finding(f"Your group found {found} of the {len(seeded)} gaps to investigate and sent {wasted} ordinary gaps to "
               f"investigation. Country A is low in chapters {', '.join(sorted(chapters))}; you ticked "
               f"{', '.join(sorted(ticked)) or 'none'}.")
    if "valuation" in answer:
        vb = pd.DataFrame(answer["valuation"]).set_index("ref")
        vb["your decision"] = vb.index.map(lambda r: LOCKED_B.get(r, "") if "LOCKED_B" in globals() else "")
        nb.show(vb[["neighbours", "share_of_median", "distance", "correct", "your decision"]], formats={"share_of_median": "{:.0%}", "distance": "{:.1f}"})
    for item in answer["explanations"]:
        print(f"- {item}")

## 6 · For your analysts

**Mirror data from UN Comtrade.** Download by hand from the Comtrade website: choose the partner as
reporter, your country as partner, exports, HS 6-digit codes and the years, then export the CSV. Keep
the reporter's FOB values and net weights; flag rows whose weight is estimated from value (Comtrade marks
them). Map HS 6 to your AHTN 8 by the first six digits before comparing.

**Regional medians.** For each good, seller and year, take the median value per kilo across the
importing countries, then each country's ratio to it. Trust a ratio only when weights cover most of the
value (here 70%): a ratio built on estimated weights says little. The sector map is the same ratio,
weighted by value, rolled up to the chapter.

**Series rules.** Persistent, growing, spike and normal come from three simple rules on the ratio after
shipping, timing and hubs: the level, the slope across years and the largest single-year drop. Read the
notes first: partner recoding, estimated weights, exempt goods and hub re-exports explain most gaps.

**Segments.** k-means puts every trader in a tile even when the groups are not clear; change k and the
seed and keep only tiles that survive. The broker signal is a share of a broker's clients with a finding:
useful for a conversation with the broker, never a verdict on one client.

**Networks.** Nodes are traders, sellers, brokers, consignees and addresses; an edge is a shared field
in the declarations. Degree (links per node) finds hubs; community detection finds groups.

**Border performance.** Time to release by channel and hit rate by channel are the two numbers to
publish monthly, beside the regime checks above.

In [ ]:
#@title 6 · The core computations
ratios = trade.mirror_ratios(mirror)                          # raw, after CIF/FOB, after timing and hubs
print(trade.classify_series(mirror).query("shape != 'normal'").to_string(index=False))
print(trade.regional_flags(ruv).to_string(index=False))
print(border.broker_scorecard(d).query("clients >= 5").head(3).to_string())